# 00 · Quickstart：10 分钟产品入口

> 本 notebook 演示：一份 A 股日频 CSV，**不写任何清洗代码**，通过 5 个 API 调用 + 若干次交互，产出一份带完整质量护照的干净 CSV。
> 它承诺：决策可追溯、差异可见、流程可复现。
> 它不承诺：自动得到正确答案。跳过打标是合法终态。

**目标读者**：新用户、评估者　　**前置条件**：`pip install -e .` 后 `import finpass`

**输入**：`examples/pingan_bank_daily_dirty.csv`（8 行，预置 5 类脏数据）　　**输出**：`out/00/`（7 个文件）

**本 notebook 的 8 个 cell**

1. 承诺与边界（本页）
2. 安装与导入
3. `fp.open`：打开数据，查看 `profile_info`
4. `session.start`：检测，返回问题队列统计
5. `session.next` / `session.answer`：一次一问
6. `session.export`：导出 7 个文件并内嵌 diff
7. 自检：断言产物、颜色、护照字段、事件与变更
8. P0 验收清单 + 下一步

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import IFrame, display

import finpass as fp

print(fp.__version__)

0.0.1


In [2]:
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
session = fp.open(ROOT / 'examples' / 'pingan_bank_daily_dirty.csv', profile='a_stock_daily')
display(session.profile_info)

{'profile': 'a_stock_daily',
 'version': '0.1.0',
 'key': '日期',
 'keys': ['日期'],
 'rules': 5,
 'rule_ids': ['missing_token',
  'duplicate_key',
  'ohlc_contradiction',
  'unit_ambiguity',
  'date_format'],
 'source': 'D:\\Files\\project\\finpass\\src\\finpass\\profiles\\a_stock_daily.yaml',
 'rows': 8,
 'cols': 7,
 'columns': ['日期', '开盘价', '最高价', '最低价', '收盘价', '成交量_股', '成交额_元']}

In [3]:
queue_summary = session.start()
display(queue_summary)
print('共', queue_summary['total_questions'], '题；命中规则', queue_summary['rules_hit'])

{'total_questions': 6,
 'strategy': 1,
 'column': 4,
 'point': 1,
 'total_affected_rows': 6,
 'rules_hit': ['date_format',
  'duplicate_key',
  'missing_token',
  'ohlc_contradiction',
  'unit_ambiguity'],
 'unanswered_count': 0}

共 6 题；命中规则 ['date_format', 'duplicate_key', 'missing_token', 'ohlc_contradiction', 'unit_ambiguity']


In [4]:
# 实际使用时，用户在 widget 里点选，render 内部调 answer；
# notebook 无人交互，因此用 DEMO_ANSWERS 按 rule_id 给示范答案。
DEMO_ANSWERS = {
    'missing_token': ('answer', {'fill': 'NaN'}, '缺失 token 统一填 NaN'),
    'duplicate_key': ('answer', {'keep': '保留第一条'}, '重复主键保留第一条'),
    'date_format': ('answer', {'fmt': '%Y-%m-%d'}, '日期统一为 %Y-%m-%d'),
    'ohlc_contradiction': ('skip', {}, '疑似数据源错误，保留打标'),
    'unit_ambiguity': ('batch_apply', {'scale': 0.0001, 'unit': '万元'}, '成交额异常行按万元折算'),
}

answered = skipped = 0
while (q := session.next()) is not None:
    display(q.render())
    action, params, reason = DEMO_ANSWERS[q.rule_id]
    session.answer(q.question_id, action, params, reason)
    answered += 1
    skipped += int(action == 'skip')

print('已答', answered, '题，跳过', skipped, '题')

'┌─ [strategy] duplicate_key  (q_dup_key)\n│ 影响行数：1    预估变更：1\n│ 样本：日期=2024-03-14\n│ 样本：日期=2024-03-14\n│ 可选答案：保留第一条 / 保留最后一条 / 跳过打标\n│ 默认建议：保留第一条\n└──────────────────────────────────────────────────────────'

'┌─ [column] missing_token  (q_missing_收盘价)\n│ 影响行数：1    预估变更：1\n│ 样本：日期=2024-03-12  收盘价=--\n│ 可选答案：填 NaN / 跳过打标\n│ 默认建议：填 NaN\n└──────────────────────────────────────────────────────────'

'┌─ [column] missing_token  (q_missing_成交额_元)\n│ 影响行数：1    预估变更：1\n│ 样本：日期=2024-03-13  成交额_元=暂无\n│ 可选答案：填 NaN / 跳过打标\n│ 默认建议：填 NaN\n└──────────────────────────────────────────────────────────'

'┌─ [column] unit_ambiguity  (q_unit_成交额_元)\n│ 影响行数：1    预估变更：1\n│ 样本：日期=2024-03-19  成交额_元=12000000\n│ 可选答案：按万元折算异常行 / 跳过打标\n│ 默认建议：按万元折算异常行\n└──────────────────────────────────────────────────────────'

'┌─ [column] date_format  (q_date_日期)\n│ 影响行数：1    预估变更：1\n│ 样本：日期=2024/03/18\n│ 可选答案：转换为 %Y-%m-%d / 跳过打标\n│ 默认建议：转换为 %Y-%m-%d\n└──────────────────────────────────────────────────────────'

'┌─ [point] ohlc_contradiction  (q_ohlc_2024-03-15)\n│ 影响行数：1    预估变更：1\n│ 样本：日期=2024-03-15  最高价=9.8  最低价=10.3  收盘价=10.4\n│ 可选答案：标记跳过 / 以收盘价为准修正\n│ 默认建议：标记跳过\n└──────────────────────────────────────────────────────────'

已答 6 题，跳过 1 题


In [5]:
OUT00 = ROOT / 'out' / '00'
session.export(OUT00)
print(sorted(p.name for p in OUT00.iterdir()))

passport = json.loads((OUT00 / 'pingan_bank_daily_dirty.passport.json').read_text(encoding='utf-8'))
display(passport)

IFrame('out/00/pingan_bank_daily_dirty.diff.html', width='100%', height=420)

['pingan_bank_daily_dirty.changes.parquet', 'pingan_bank_daily_dirty.clean.csv', 'pingan_bank_daily_dirty.diff.html', 'pingan_bank_daily_dirty.events.jsonl', 'pingan_bank_daily_dirty.passport.json', 'pingan_bank_daily_dirty.report.after.json', 'pingan_bank_daily_dirty.report.before.json']


{'input_hash': '4848987fd8e9c828',
 'output_file_sha256': 'e8ab74d7cf49bce0363c4105473b89a9be65454628f0387f8d678e2acab39bc5',
 'input_path': 'd:\\Files\\project\\finpass\\examples\\pingan_bank_daily_dirty.csv',
 'keys': ['日期'],
 'profile_name': 'a_stock_daily',
 'profile_version': '0.1.0',
 'human_decisions_count': 6,
 'unresolved_issues_count': 4,
 'auto_only': False,
 'cli_replay_cmd': "'d:\\Files\\project\\finpass\\.venv\\Scripts\\python.exe' -m finpass run 'd:\\Files\\project\\finpass\\examples\\pingan_bank_daily_dirty.csv' --profile a_stock_daily --interactive cli --out 'd:\\Files\\project\\finpass\\out\\00'",
 'reproduce': 'finpass verify pingan_bank_daily_dirty.passport.json'}

In [6]:
OUT00 = ROOT / 'out' / '00'
STEM = 'pingan_bank_daily_dirty'

names = ['clean.csv', 'passport.json', 'events.jsonl', 'changes.parquet',
         'report.before.json', 'report.after.json', 'diff.html']
missing = [n for n in names if not (OUT00 / f'{STEM}.{n}').exists()]
assert not missing, f'缺少产物：{missing}'
assert len(list(OUT00.iterdir())) == 7, list(OUT00.iterdir())

html = (OUT00 / f'{STEM}.diff.html').read_text(encoding='utf-8')
colors = [c for c in ['#d4f8d4', '#fff3cd', '#d1ecf1', '#f8d7da', '#ffe8b3'] if c in html]
assert len(colors) >= 4, colors
for must in ['#d4f8d4', '#f8d7da', '#ffe8b3']:
    assert must in html, f'缺少颜色 {must}'

passport = json.loads((OUT00 / f'{STEM}.passport.json').read_text(encoding='utf-8'))
for field in ['input_hash', 'output_file_sha256', 'human_decisions_count',
              'unresolved_issues_count', 'auto_only']:
    assert field in passport, field

events = (OUT00 / f'{STEM}.events.jsonl').read_text(encoding='utf-8').strip().splitlines()
assert len(events) == passport['human_decisions_count'], (len(events), passport['human_decisions_count'])

changes = pd.read_parquet(OUT00 / f'{STEM}.changes.parquet')
assert len(changes) > 0
assert changes['decision_id'].notna().all()

print('✅ 00 验收通过', '| 颜色', colors, '| 事件', len(events), '| 变更', len(changes))

✅ 00 验收通过 | 颜色 ['#d4f8d4', '#fff3cd', '#d1ecf1', '#f8d7da', '#ffe8b3'] | 事件 6 | 变更 9


## P0 验收清单与下一步

**P0 硬标准**

- [ ] 用户全程只写 `fp.open` / `session.start` / `session.next` / `session.answer` / `session.export`，不写任何清洗逻辑；
- [ ] 5 类问题全部触发（缺失 token、重复行、OHLC 矛盾、单位歧义、日期格式）；
- [ ] `out/00/` 下 7 个文件齐全；
- [ ] `diff.html` 独立打开，≥4 种颜色，hover 有 `rule_id + decision_id`；
- [ ] `passport.json` 能回答：输入 hash、profile 版本、人工决策数、未解决数；
- [ ] 无 widget 环境下 `q.render()` 退化为纯文本卡片，流程仍能跑完。

**已达成**（Cell 7 自检）：7 个产物、≥4 色、护照 5 个字段、事件数 = 决策数、`changes` 每行带 `decision_id`。

**下一步**

- 想看内核怎么实现的 → `01_minimal_kernel.ipynb`
- 想看问题聚合 → `02_interaction_queue.ipynb`（P1）
- 想新增数据域 → `04_profile_authoring.ipynb`（P2）